In [ ]:
# create the directory structure
!mkdir -p MLChurn/data/raw
!mkdir -p MLChurn/data/procesed
!mkdir -p MLChurn/src
# MLChurn/data/raw - capture the raw data
# MLChurn/data/procesed - store clean data
# MLChurn/src - store code files

# create empty files
!touch MLChurn/src/ingest.py
!touch MLChurn/src/preprocess.py
!touch MLChurn/src/train.py

In [ ]:
!ls -R MLChurn

MLChurn:
data  src

MLChurn/data:
procesed  raw

MLChurn/data/procesed:

MLChurn/data/raw:

MLChurn/src:
ingest.py  preprocess.py  train.py


In [ ]:
!touch MLChurn/requirements.txt

In [ ]:
%%writefile /content/MLChurn/requirements.txt
mlflow
pyngrok

Overwriting /content/MLChurn/requirements.txt


In [ ]:
!cat MLChurn/requirements.txt

mlflow
pyngrok


In [ ]:
%%writefile /content/MLChurn/src/ingest.py

# to collect the data and store it in our location for further use
def ingest_data():
  import pandas as pd
  data_src = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
  data_dest = "/content/MLChurn/data/raw/"

  df = pd.read_csv(data_src)
  #print(df.head(2))
  df.to_csv(data_dest+'data.csv',index=False)
  print('Data ingestion completed. check folder /content/MLChurn/data/raw')

if __name__ == "__main__":
  ingest_data()

Overwriting /content/MLChurn/src/ingest.py


In [ ]:
!python MLChurn/src/ingest.py

In [ ]:
%%writefile /content/MLChurn/src/preprocess.py

def preprocess_data():
  # set the source and destination path
  import pandas as pd
  data_src = '/content/MLChurn/data/raw'
  data_dest = '/content/MLChurn/data/procesed'

  # load data from source
  df = pd.read_csv(data_src+'/data.csv')
  #print(df.shape)
  #print(df.info())
  #print(df.head(1))

  # to mark the features to ignore for preprocess
  ign_cols = ['customerID','Churn']
  #print(ign_cols)

  # find the category and numerical columns
  cat_cols = df.drop(ign_cols,axis=1).select_dtypes(include='object').columns
  #print(cat_cols)
  num_cols = df.drop(ign_cols,axis=1).select_dtypes(exclude='object').columns
  #print(num_cols)

  #print(len(cat_cols), len(num_cols))

  # check null value and fix it using mean for numeric value and mode for categorical value
  if df[cat_cols].isnull().sum().any():
    df[cat_cols].fillna(df[cat_cols].mode())

  if df[num_cols].isnull().sum().any():
    df[num_cols].fillna(df[num_cols].mean())

  # check and convert target value to numerical
  #print(df['Churn'].unique())
  df['Churn'] = df['Churn'].map({'No':0,'Yes':1})
  #print(df['Churn'].unique())

  # do category encoding
  df_enc = pd.get_dummies(df[cat_cols],columns=cat_cols,drop_first=True,dtype=int)
  #print(df_enc.shape)
  #print(df_enc.head(1))
  #print(df_enc.info())

  # concat all the non-encoded(num_cols)+tgt_cols
  df_new = pd.concat([df[num_cols],df_enc,df['Churn']],axis=1)

  df_new.to_csv(data_dest+'/preprcessed_data.csv',index=False)
  print('Data preprocessing completed.  check folder /content/MLChurn/data/procesed')

if __name__ == "__main__":
  preprocess_data()

Overwriting /content/MLChurn/src/preprocess.py


In [ ]:
%%writefile /content/MLChurn/src/train.py

def train_model():
  import mlflow
  import sqlite3
  import pandas as pd
  from sklearn.model_selection import train_test_split
  from sklearn.linear_model import LogisticRegression
  from sklearn.metrics import accuracy_score, f1_score
  from sklearn.tree import DecisionTreeClassifier

  # store tracking info in a light weight db called sqlite
  MLFLOW_TRACKING_URI = 'sqlite:///myprojflow.db'
  mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
  experiment = mlflow.set_experiment('Customer churn Classification')
  print(experiment.name)

  #experiment = mlflow.set_experiment('Customer churn Classification')
  #print(experiment.name)
  print(experiment.experiment_id)

  data_src = '/content/MLChurn/data/procesed/'
  df = pd.read_csv(data_src+'preprcessed_data.csv',nrows=100)

  print(df.shape)
  print(df.head(1))
  #print(df.info())

  X = df.drop('Churn',axis=1)
  y = df['Churn']

  X_train,X_val,y_train,y_val=train_test_split(X,y,test_size=0.2,random_state=42)
  print(X_train.shape,X_val.shape,y_train.shape,y_val.shape)

  run_name = 'LR L2'
  lm_name = 'lm '+run_name
  with mlflow.start_run(experiment_id = experiment.experiment_id,run_name=run_name ):
    #model = LogisticRegression(penalty='l2', solver='lbfgs',max_iter=90)
    #model = LogisticRegression(penalty='l1',solver='saga',max_iter=50)

    model.fit(X_train,y_train)

    pred = model.predict(X_val)
    acc = accuracy_score(y_val,pred)
    f1 = f1_score(y_val,pred)

    #print(acc)
    #print(f1)
    mlflow.log_param('model',run_name)
    print(model)
    if isinstance(model, DecisionTreeClassifier):
      mlflow.log_param('max_depth',model.max_depth)
    elif isinstance(model, LogisticRegression):
      mlflow.log_param('penalty',model.penalty)
      mlflow.log_param('max_iter',model.max_iter)

    mlflow.log_metric('f1_score',f1)
    mlflow.log_metric('acc_score',acc)
    mlflow.sklearn.log_model(model,"model")
if __name__ == "__main__":
  train_model()


Overwriting /content/MLChurn/src/train.py


In [ ]:
!pip install -r /content/MLChurn/requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 85.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 789.2/789.2 kB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 15.6 MB/s eta 0:00:00


In [ ]:
import mlflow
# store tracking info in a light weight db called sqlite
MLFLOW_TRACKING_URI = 'sqlite:///myprojflow.db'
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
experiment = mlflow.set_experiment('Customer churn Classification')
print(experiment.name)


2026/02/01 03:26:22 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/02/01 03:26:22 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/02/01 03:26:22 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/02/01 03:26:22 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/02/01 03:26:22 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/02/01 03:26:22 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/02/01 03:26:22 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/02/01 03:26:22 INFO mlflow.store.db.utils: Updating database tables
2026/02/01 03:26:22 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/02/01 03:26:22 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2026/02/01 03:26:23 INFO alembic.runtime.migration: Running upgrade  -> 451aebb31d03, add metric step
2026/02/01 03:2

Customer churn Classification


In [ ]:
import sys
sys.path.append('/content/MLChurn/src') # /content/MLChurn/src
#from MLChurn.src.ingest import ingest_data
import ingest
import preprocess
import train

# to reload the model into current session memory
import importlib
importlib.reload(ingest)
importlib.reload(preprocess)
importlib.reload(train)

# calling/using the method from the module
ingest.ingest_data()
preprocess.preprocess_data()
train.train_model()

Data ingestion completed. check folder /content/MLChurn/data/raw
Data preprocessing completed.  check folder /content/MLChurn/data/procesed
Customer churn Classification
1
(100, 6560)
   SeniorCitizen  tenure  MonthlyCharges  gender_Male  Partner_Yes  \
0              0       1           29.85            0            1   

   Dependents_Yes  PhoneService_Yes  MultipleLines_No phone service  \
0               0                 0                               1   

   MultipleLines_Yes  InternetService_Fiber optic  ...  TotalCharges_996.45  \
0                  0                            0  ...                    0   

   TotalCharges_996.85  TotalCharges_996.95  TotalCharges_997.65  \
0                    0                    0                    0   

   TotalCharges_997.75  TotalCharges_998.1  TotalCharges_999.45  \
0                    0                   0                    0   

   TotalCharges_999.8  TotalCharges_999.9  Churn  
0                   0                   0      0  

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
2026/02/01 03:27:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


LogisticRegression(max_iter=90)


/usr/local/lib/python3.12/dist-packages/mlflow/models/model.py:1209: FutureWarning: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.
  flavor.save_model(path=local_path, mlflow_model=mlflow_model, **kwargs)


In [ ]:
!python MLChurn/src/train.py

2026/02/01 03:39:12 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/02/01 03:39:12 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/02/01 03:39:12 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/02/01 03:39:12 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/02/01 03:39:12 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/02/01 03:39:12 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/02/01 03:39:13 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/02/01 03:39:13 INFO alembic.runtime.migration: Will assume non-transactional DDL.
Customer churn Classification
1
(100, 6560)
   SeniorCitizen  tenure  ...  TotalCharges_999.9  Churn
0              0       1  ...                   0      0

[1 rows x 6560 columns]
(80, 6559) (20, 6559) (80,) (20,)
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sa

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('myprojflow.db')

e = pd.read_sql_query('SELECT * FROM experiments',conn)
display(e)

m = pd.read_sql_query('SELECT * FROM metrics',conn)
display(m)

p = pd.read_sql_query('SELECT * FROM params',conn)
display(p)

lm = pd.read_sql_query("SELECT * FROM logged_models;", conn)
display(lm)

summ = pd.read_sql_query('''
  SELECT e.name, m.key, MAX(m.value), p.key, p.value, r.name, r.run_uuid
  FROM experiments e
  JOIN runs r on e.experiment_id = r.experiment_id
  JOIN metrics m on r.run_uuid = m.run_uuid AND m.key='f1_score'
  JOIN params p on r.run_uuid = p.run_uuid
  --WHERE #e.experiment_id = m.experiment_id AND e.experiment_id = p.experiment_id
  ''',conn)
display(summ)

summ = pd.read_sql_query('''
  SELECT e.name, m.key, MAX(m.value), p.key, p.value, r.name, r.run_uuid
  FROM experiments e
  JOIN runs r on e.experiment_id = r.experiment_id
  JOIN metrics m on r.run_uuid = m.run_uuid AND m.key='acc_score'
  JOIN params p on r.run_uuid = p.run_uuid
  --WHERE #e.experiment_id = m.experiment_id AND e.experiment_id = p.experiment_id
  ''',conn)

display(summ)

,experiment_id,name,artifact_location,lifecycle_stage,creation_time,last_update_time
0,0,Default,/content/mlruns/0,active,1769916385547,1769916385547
1,1,Customer churn Classification,/content/mlruns/1,active,1769916385566,1769916385566


,key,value,timestamp,run_uuid,step,is_nan
0,f1_score,0.666667,1769916464168,9832f0ac91a34a6db05d027b1a010c69,0,0
1,acc_score,0.800000,1769916464184,9832f0ac91a34a6db05d027b1a010c69,0,0
2,f1_score,0.666667,1769917154288,95b90ef535944d97884a39285c4dcd5e,0,0
3,acc_score,0.700000,1769917154305,95b90ef535944d97884a39285c4dcd5e,0,0


,key,value,run_uuid
0,model,LR L2,9832f0ac91a34a6db05d027b1a010c69
1,penalty,l2,9832f0ac91a34a6db05d027b1a010c69
2,max_iter,90,9832f0ac91a34a6db05d027b1a010c69
3,model,LR L2,95b90ef535944d97884a39285c4dcd5e
4,penalty,l1,95b90ef535944d97884a39285c4dcd5e
5,max_iter,50,95b90ef535944d97884a39285c4dcd5e


,model_id,experiment_id,name,artifact_location,creation_timestamp_ms,last_updated_timestamp_ms,status,lifecycle_stage,model_type,source_run_id,status_message
0,m-adf846d2e44b41afb8ed480ae277bc8a,1,model,/content/mlruns/1/models/m-adf846d2e44b41afb8e...,1769916464223,1769916471895,2,active,None,9832f0ac91a34a6db05d027b1a010c69,None
1,m-c588f44a0707471a9cc1befcc9303ba7,1,model,/content/mlruns/1/models/m-c588f44a0707471a9cc...,1769917154348,1769917160848,2,active,None,95b90ef535944d97884a39285c4dcd5e,None


,name,key,MAX(m.value),key,value,name,run_uuid
0,Customer churn Classification,f1_score,0.666667,model,LR L2,LR L2,9832f0ac91a34a6db05d027b1a010c69


,name,key,MAX(m.value),key,value,name,run_uuid
0,Customer churn Classification,acc_score,0.8,model,LR L2,LR L2,9832f0ac91a34a6db05d027b1a010c69
